# Setup (Colab)
1. Get Gemini API key: [ai.google.dev](https://ai.google.dev)
2. Replace `YOUR_GEMINI_API_KEY_HERE`
3. Runtime → Run all

In [ ]:
!pip -q install rdkit google-generativeai pandas numpy rich tabulate

In [ ]:
import os, json, textwrap, math, re
from google import genai
!pip -q install -U google-genai

os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE" # Add your key here
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [ ]:
#----------------Imports + API key--------------------------
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, QED, Crippen, rdMolDescriptors
from rdkit.Chem.rdMolDescriptors import CalcNumRotatableBonds
from rdkit.Chem import Draw
from rich.console import Console
from rich.table import Table

console = Console()

MODEL_NAME = "gemini-2.5-flash"

In [ ]:
#------------------Functions----------------------------
def mol_from_smiles(smiles):
    m = Chem.MolFromSmiles(smiles)
    return m


def calc_properties(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    rotb = CalcNumRotatableBonds(mol)
    rings = rdMolDescriptors.CalcNumRings(mol)
    qed = QED.qed(mol)

    lipinski_violations = int(mw > 500) + int(logp > 5) + int(hbd > 5) + int(hba > 10)
    lipinski_pass = lipinski_violations <= 1

    # FIXED heuristic (0–100)
    score = 100

    # MW: optimal 200–450
    if mw < 200: score -= (200 - mw) * 0.1
    elif mw > 500: score -= (mw - 500) * 0.15

    # logP: optimal 1.5–4.0
    if logp < 0: score -= abs(logp) * 20
    elif logp < 1.5: score -= (1.5 - logp) * 10
    elif logp > 5: score -= (logp - 5) * 8

    # TPSA: optimal < 140
    if tpsa > 140: score -= (tpsa - 140) * 0.2

    # HBD/HBA: penalize excessive
    score -= hbd * 3 if hbd > 3 else 0
    score -= (hba - 6) * 1.5 if hba > 6 else 0

    # Rotatable bonds: penalize > 8
    if rotb > 8: score -= (rotb - 8) * 3

    # Reward QED but capped
    score += min(qed * 20, 15)

    score = max(0, min(100, score))

    return {
        "smiles": smiles,
        "mw": round(mw, 2), "logP": round(logp, 2), "tpsa": round(tpsa, 2),
        "hbd": int(hbd), "hba": int(hba), "rotb": int(rotb), "rings": int(rings),
        "qed": round(float(qed), 3), "lipinski_pass": bool(lipinski_pass),
        "lipinski_violations": int(lipinski_violations),
        "heuristic_admet_score": round(float(score), 1),
    }

rows = []
for item in molecules:
    p = calc_properties(item["smiles"])
    if p:
        p["name"] = item["name"]
        rows.append(p)

df = pd.DataFrame(rows)


#--------------------------Ranking-----------------------------
df["rank"] = df["heuristic_admet_score"].rank(ascending=False, method="min").astype(int)
df = df.sort_values(["heuristic_admet_score", "qed"], ascending=[False, False]).reset_index(drop=True)


#--------------------------Gemini agent functions-----------------
def gemini_json(prompt):
    """Send prompt to Gemini and parse JSON (or fallback)."""
    resp = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
    )
    txt = resp.text.strip()
    txt = re.sub(r"^```json\s*|\s*```$", "", txt, flags=re.S)
    try:
        return json.loads(txt)
    except Exception:
        return {"raw": txt}

def interpret_molecule(row):
    prompt = f"""
You are an expert medicinal chemist reviewing ADMET-like triage output.

Return STRICT JSON with keys:
- summary: <= 35 words
- liabilities: array of short strings
- optimization: array of 2-4 short molecule-edit suggestions
- confidence: one of ["low","medium","high"]

Molecule:
Name: {row['name']}
SMILES: {row['smiles']}
MW: {row['mw']}
logP: {row['logP']}
TPSA: {row['tpsa']}
HBD: {row['hbd']}
HBA: {row['hba']}
Rotatable bonds: {row['rotb']}
Rings: {row['rings']}
QED: {row['qed']}
Lipinski pass: {row['lipinski_pass']}
Lipinski violations: {row['lipinski_violations']}
Heuristic score: {row['heuristic_admet_score']}

Be biologically sensible. Do not invent experimental data.
"""
    return gemini_json(prompt)

def propose_optimized_smiles(row, advice):
    return row["smiles"]

agent_outputs = []
for _, row in df.iterrows():
    out = interpret_molecule(row)
    agent_outputs.append(out)

df["agent_summary"] = [x.get("summary", x.get("raw", "")) for x in agent_outputs]
df["agent_liabilities"] = [", ".join(x.get("liabilities", [])) for x in agent_outputs]
df["agent_optimization"] = [", ".join(x.get("optimization", [])) for x in agent_outputs]
df["agent_confidence"] = [x.get("confidence", "unknown") for x in agent_outputs]

In [ ]:
molecules_step1 = [
    {"name": "Aspirin",   "smiles": "CC(=O)Oc1ccccc1C(=O)O"},
    {"name": "Ibuprofen", "smiles": "CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O"},
]

# Run RDKit + agent loop
rows = []
for item in molecules_step1:
    p = calc_properties(item["smiles"])
    if p:
        p["name"] = item["name"]
        rows.append(p)

df1 = pd.DataFrame(rows)
df1["rank"] = df1["heuristic_admet_score"].rank(ascending=False, method="min").astype(int)
df1 = df1.sort_values(["heuristic_admet_score"], ascending=[False]).reset_index(drop=True)

agent_outputs = []
for _, row in df1.iterrows():
    out = interpret_molecule(row)
    agent_outputs.append(out)

df1["agent_summary"] = [x.get("summary", x.get("raw", "")) for x in agent_outputs]
df1["agent_liabilities"] = [", ".join(x.get("liabilities", [])) for x in agent_outputs]
df1["agent_optimization"] = [", ".join(x.get("optimization", [])) for x in agent_outputs]

display_cols = ["name","mw","logP","tpsa","hbd","hba","rotb","qed","lipinski_pass","heuristic_admet_score","agent_summary"]
print("\n" + "="*80)
print("STEP 1: Baseline validation (good ADMET drugs)")
print("="*80)
print(df1[display_cols])

In [ ]:
molecules_step2 = [
    {"name": "Metformin", "smiles": "CN(C)C(=N)N"},
    {"name": "Lopinavir","smiles": "CC(C)(C)C(=O)N[C@@H](C(=O)N1C[C@@H](C[C@H]1C(=O)N[C@H](C(=O)N[C@H](C(=O)OC)C(C)(C)C)Cc2ccccc2)C(=O)N[C@H](C(=O)O)C(C)(C)C)C(=O)OC"},
]

# Run RDKit + agent loop
rows = []
for item in molecules_step2:
    p = calc_properties(item["smiles"])
    if p:
        p["name"] = item["name"]
        rows.append(p)

df2 = pd.DataFrame(rows)
df2["rank"] = df2["heuristic_admet_score"].rank(ascending=False, method="min").astype(int)
df2 = df2.sort_values(["heuristic_admet_score"], ascending=[False]).reset_index(drop=True)

agent_outputs = []
for _, row in df2.iterrows():
    out = interpret_molecule(row)
    agent_outputs.append(out)

df2["agent_summary"] = [x.get("summary", x.get("raw", "")) for x in agent_outputs]
df2["agent_liabilities"] = [", ".join(x.get("liabilities", [])) for x in agent_outputs]
df2["agent_optimization"] = [", ".join(x.get("optimization", [])) for x in agent_outputs]

print("\n" + "="*80)
print("STEP 2: Tradeoff analysis (Metformin vs Lopinavir)")
print("="*80)
print(df2[["name","mw","logP","tpsa","hbd","hba","rotb","qed","lipinski_pass","heuristic_admet_score","agent_summary"]])

In [ ]:
molecules_step3 = [
    {"name": "Sucrose",      "smiles": "C([C@@H]1[C@H]([C@@H]([C@H]([C@@H]([C@H]1O)O)O)O)CO)O"},
    {"name": "Stearic acid", "smiles": "CCCCCCCCCCCCCCCCCC(=O)O"},
]

# Run RDKit + agent loop
rows = []
for item in molecules_step3:
    p = calc_properties(item["smiles"])
    if p:
        p["name"] = item["name"]
        rows.append(p)

df3 = pd.DataFrame(rows)
df3["rank"] = df3["heuristic_admet_score"].rank(ascending=False, method="min").astype(int)
df3 = df3.sort_values(["heuristic_admet_score"], ascending=[False]).reset_index(drop=True)

agent_outputs = []
for _, row in df3.iterrows():
    out = interpret_molecule(row)
    agent_outputs.append(out)

df3["agent_summary"] = [x.get("summary", x.get("raw", "")) for x in agent_outputs]
df3["agent_liabilities"] = [", ".join(x.get("liabilities", [])) for x in agent_outputs]
df3["agent_optimization"] = [", ".join(x.get("optimization", [])) for x in agent_outputs]

print("\n" + "="*80)
print("STEP 3: Failure cases (non‑drug‑like molecules)")
print("="*80)
print(df3[["name","mw","logP","tpsa","hbd","hba","rotb","qed","lipinski_pass","heuristic_admet_score","agent_summary"]])

In [ ]:
molecules_step4 = [
    {"name": "Imatinib",    "smiles": "CN(C)CCNC(=N)N1C([C@H](C([C@H]2C3=CC=CC=C3C(=C2)C)N4C=[CH]C(=C4)C)C(=N1)C5=CC=CC=C5)C(=O)NC6=CC=CC=C6"},
    {"name": "Chloroquine", "smiles": "CN(C)CCN(CCCl)C1=CC=CC=C1Cl"},
]

# Run RDKit + agent loop
rows = []
for item in molecules_step4:
    p = calc_properties(item["smiles"])
    if p:
        p["name"] = item["name"]
        rows.append(p)

df4 = pd.DataFrame(rows)
df4["rank"] = df4["heuristic_admet_score"].rank(ascending=False, method="min").astype(int)
df4 = df4.sort_values(["heuristic_admet_score"], ascending=[False]).reset_index(drop=True)

agent_outputs = []
for _, row in df4.iterrows():
    out = interpret_molecule(row)
    agent_outputs.append(out)

df4["agent_summary"] = [x.get("summary", x.get("raw", "")) for x in agent_outputs]
df4["agent_liabilities"] = [", ".join(x.get("liabilities", [])) for x in agent_outputs]
df4["agent_optimization"] = [", ".join(x.get("optimization", [])) for x in agent_outputs]

print("\n" + "="*80)
print("STEP 4: Real‑world complexity (Imatinib, Chloroquine)")
print("="*80)
print(df4[["name","mw","logP","tpsa","hbd","hba","rotb","qed","lipinski_pass","heuristic_admet_score","agent_summary"]])

In [ ]:
df1.to_csv("step1_baseline.csv")
df2.to_csv("step2_tradeoff.csv")
df3.to_csv("step3_failure.csv")
df4.to_csv("step4_realWorld.csv")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.patches as mpatches

df_all = (
    pd.concat([
        df1.assign(step="1: Baseline").reset_index(drop=True),
        df2.assign(step="2: Tradeoff").reset_index(drop=True),
        df3.assign(step="3: Failure").reset_index(drop=True),
        df4.assign(step="4: Real‑world").reset_index(drop=True),
    ], ignore_index=True)
    .sort_values(["heuristic_admet_score"], ascending=False)
)

# Core columns for plotting and table
plot_cols = ["name", "step", "heuristic_admet_score", "mw", "logP", "lipinski_pass"]
plot_df = df_all[plot_cols].round(2)

#-----------------Bar chart of ADMET score-----------------

fig, ax = plt.subplots(figsize=(10, 7))

step_order = {
    "1: Baseline": 1,
    "2: Tradeoff": 2,
    "3: Failure": 3,
    "4: Real‑world": 4,
}
plot_df["step_num"] = plot_df["step"].map(step_order)
plot_df = plot_df.sort_values(["step_num", "heuristic_admet_score"], ascending=[True, False])

colors = {
    "1: Baseline": "steelblue",
    "2: Tradeoff": "orange",
    "3: Failure": "red",
    "4: Real‑world": "green",
}

bars = ax.barh(
    range(len(plot_df)),
    plot_df["heuristic_admet_score"],
    color=[colors[s] for s in plot_df["step"]],
    edgecolor="none",
    height=0.6
)

ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(
    [f"{row['name']} ({row['step']})" for _, row in plot_df.iterrows()],
    fontsize=9
)
ax.set_xlabel("Heuristic ADMET Score (0–100)", fontsize=11)
ax.set_ylabel("Molecule (Step)", fontsize=11)
ax.set_title("Agentic ADMET Predictor: All Examples", fontsize=13, pad=15)

for i, (rect, row) in enumerate(zip(bars, plot_df.itertuples())):
    ax.text(
        rect.get_width() + 1,
        rect.get_y() + rect.get_height()/2,
        f"{row.heuristic_admet_score:.1f}",
        va="center",
        fontsize=8,
        color="gray"
    )

# Legend
legend_handles = [
    mpatches.Patch(color="steelblue", label="1: Baseline"),
    mpatches.Patch(color="orange",  label="2: Tradeoff"),
    mpatches.Patch(color="red",     label="3: Non-drug-like/Failure"),
    mpatches.Patch(color="green",   label="4: Real‑world"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="lower right")

fig.tight_layout()
plt.savefig("output/admet_all_steps.png", dpi=150, bbox_inches="tight")
plt.show()

#-----------------------Table image-----------------------
plot_df_sorted = plot_df.sort_values(["step_num", "heuristic_admet_score"], ascending=[True, False])

fig2, ax2 = plt.subplots(figsize=(9, 4))
ax2.axis("off")

table = ax2.table(
    cellText=plot_df_sorted[["name", "step", "heuristic_admet_score", "mw", "logP"]].astype(str).values,
    colLabels=["Name", "Step", "Score", "MW", "logP"],
    cellLoc="center",
    loc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.1, 1.8)

ax2.set_title("Agentic ADMET Examples Summary", fontsize=12, pad=15)
fig2.tight_layout()
plt.savefig("output/admet_table.png", dpi=150, bbox_inches="tight")
plt.show()